# MaxKernel 原 Claude Code pipeline · 原生 Codex CLI · TPU v6e

由 **Codex CLI 本身**执行模型调用、原生子 agent、Shell 与文件操作，继续使用原 MaxKernel 全部阶段 prompts、知识库和工具。Python 启动器只准备配置、启动 CLI、记录日志和处理暂停/恢复。

选择 Colab **TPU runtime**，使用独占 runtime，停止原 ADK/其他实验。默认 BF16 512×512 matmul + bias + ReLU，原流程 **5 轮**，每轮保留编译、正确性、autotune 和 profile。

已通过真实 CLI + 本机模拟 API 的离线集成测试；实际代理协议、Colab 完整依赖安装与 TPU 效果需要执行本实验验证。

## 1. 模型、CLI 和实验配置
密钥通过环境变量、Colab Secrets 或交互输入读取。CLI 固定为已验证的 `0.171.0-alpha.13`。360 秒表示 **流式响应无新事件时的超时**，不是一次模型调用的总时限。CLI 自行管理输出预算，本入口不使用 SDK 的 `MAXKERNEL_MAX_OUTPUT_TOKENS`。

In [ ]:
import os
import subprocess
import sys
from datetime import datetime
from getpass import getpass
from pathlib import Path

REPO_URL = "https://github.com/yezi333/accelerator-agents.git"
BRANCH = "codex/claude-pipeline-codex-cli"
REPO = Path("/content/maxkernel-native-codex")
ROOT = REPO / "maxkernel/claude-code"
VENV = Path("/content/maxkernel-native-codex-venv")
PYTHON = VENV / "bin/python"
CODEX_VERSION = "0.171.0-alpha.13"
CLI_INSTALL = Path("/content/maxkernel-codex-cli-bin")
CODEX = CLI_INSTALL / "node_modules/.bin/codex"
SANDBOX = "workspace-write"

RUN_NAME = "codex-cli-matmul-" + datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_DIR = ROOT / "workspace" / RUN_NAME

os.environ.update(
  {
    "OPENAI_BASE_URL": "https://agent.infiscale-tech.com/v1",
    "MAXKERNEL_MODEL": "gpt-5.6-sol",
    "MAXKERNEL_REASONING_EFFORT": "high",
    "MAXKERNEL_API_TIMEOUT": "360",
    "MAXKERNEL_API_MAX_RETRIES": "2",
  }
)
if not os.environ.get("OPENAI_API_KEY"):
  try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
  except Exception:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
assert os.environ.get("OPENAI_API_KEY"), "需要 API key"
print("Model:", os.environ["MAXKERNEL_MODEL"])
print("Run:", RUN_DIR)
# 恢复旧实验时，将 RUN_DIR 改为该实验的已有绝对路径，第 4 节会识别已有请求并跳过首次 dry-run。

## 2. 获取专用分支
已有 checkout 必须位于同一分支且无本地修改。运行途中不要重新拉取代码。

In [ ]:
def checked(args, **kwargs):
  return subprocess.run([str(x) for x in args], check=True, **kwargs)


if not REPO.exists():
  checked(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO]
  )
else:
  branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO, text=True
  ).strip()
  assert branch == BRANCH, f"已有 checkout 分支不符: {branch}"
  dirty = subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=REPO, text=True
  )
  assert not dirty.strip(), "已有本地修改，请先处理或使用新的 checkout 目录"
  checked(["git", "pull", "--ff-only"], cwd=REPO)
checked(["git", "rev-parse", "HEAD"], cwd=REPO)

## 3. 安装原 pipeline 环境与 Codex CLI
独立 Python 3.12 环境保留原 `claude-code/requirements.txt` 的 TPU 版本；不安装自建 OpenAI 宿主。安装固定 npm 包，不修改个人 Codex 配置。

默认 `workspace-write` 沙箱。如果 Colab 容器不支持 CLI 沙箱，入口会报错，不会自动放开权限。确认使用独占、可丢弃的 runtime 后，可自行把第 1 节的 `SANDBOX` 改为 `danger-full-access`，再重新生成第 4 节的启动命令；此时 CLI 命令可操作整个 runtime。原 Claude 可选 guard hook 没有移植，不能视为同等权限隔离。

In [ ]:
import shutil

if not shutil.which("node") or not shutil.which("npm"):
  checked(["apt-get", "update", "-qq"])
  checked(["apt-get", "install", "-y", "nodejs", "npm"])
checked(
  ["npm", "install", "--prefix", CLI_INSTALL, "@openai/codex@" + CODEX_VERSION]
)
checked([CODEX, "--version"])
checked([sys.executable, "-m", "pip", "install", "uv"])
if not PYTHON.exists():
  checked([sys.executable, "-m", "uv", "venv", "--python", "3.12", VENV])
checked(
  [
    sys.executable,
    "-m",
    "uv",
    "pip",
    "install",
    "--python",
    PYTHON,
    "-r",
    ROOT / "requirements.txt",
  ]
)
checked([sys.executable, "-m", "uv", "pip", "check", "--python", PYTHON])
checked(
  [
    PYTHON,
    "-c",
    "import jax, jaxlib; print('JAX', jax.__version__, 'jaxlib', jaxlib.__version__); print(jax.devices()); assert any(d.platform == 'tpu' for d in jax.devices()), '请选择 TPU runtime'",
  ],
  cwd=ROOT,
)
# TPU 探测在独立进程执行并退出，避免 notebook 主进程持有 TPU。

## 4. 选择 kernel，验证原生角色注册
默认共用上一方案的同一个 JAX 示例，以便比较。自己的任务可填写原格式 job.json 或自包含 .py。预检查调用 CLI 的本地 prompt 调试入口，核对全部原生角色，**不调用 API/TPU**。已有请求时保留原输入，稍后按磁盘 state 恢复。

In [ ]:
JOB_PATH = ROOT / "openai_runtime/examples/matmul_bias_relu.job.json"
SOURCE_PATH = None  # 例如 Path("/content/my_kernel.py")；设置后替代 JOB_PATH
BASE_CMD = [
  str(PYTHON),
  "-u",
  "-m",
  "codex_cli",
  "--run-dir",
  str(RUN_DIR),
  "--python",
  str(PYTHON),
  "--codex",
  str(CODEX),
  "--sandbox",
  SANDBOX,
  "--tpu-version",
  "TPU v6e",
]
input_args = (
  ["--source", str(SOURCE_PATH)] if SOURCE_PATH else ["--job", str(JOB_PATH)]
)
if (RUN_DIR / "codex_host/request.json").exists():
  print("已存在请求，将用 --resume 恢复；不替换原输入。")
else:
  checked(BASE_CMD + input_args + ["--dry-run"], cwd=ROOT)

## 5. 启动原生 Codex CLI 完整流程
本 cell 开始消耗模型 API 与 TPU。Python 启动器调用 `codex exec`，由 Codex 原生创建 worker 和 specialists 并执行工具。

退出码 2 表示待人工决定；先阅读第 6 节问题及产物，再执行第 7 节。中断后可在同一个 RUN_DIR 上重跑本 cell：新的 CLI 根会话读取磁盘 state，未完成阶段可能重做。不要在运行中更新 checkout 或重新安装环境。

In [ ]:
import signal


def run_pipeline(extra_args=None):
  command = BASE_CMD + ["--resume"] + (extra_args or [])
  process = subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    start_new_session=True,
  )
  try:
    for line in process.stdout:
      print(line, end="", flush=True)
    returncode = process.wait()
  except KeyboardInterrupt:
    # SIGINT 让 Python 宿主保存状态并清理正在运行的工具进程。
    if process.poll() is None:
      os.killpg(process.pid, signal.SIGINT)
    try:
      process.wait(timeout=30)
    except subprocess.TimeoutExpired:
      os.killpg(process.pid, signal.SIGKILL)
      process.wait()
    print("已中断；请查看状态后恢复。原 TPU server/已排队 job 可能仍在运行。")
    return 130
  finally:
    process.stdout.close()
  print("Exit code:", returncode)
  if returncode == 2:
    print("等待人工输入：先查看 pending_question.json，再填写回复。")
  elif returncode != 0:
    print("本次未完整完成；查看 status.json 和 events.jsonl 后决定是否恢复。")
  return returncode


exit_code = run_pipeline()

## 6. 检查状态和结果
`completed` 表示轮数已跑完，不保证生成了正确或更快的 kernel。结合 `has_valid_candidate`、每轮正确性和实测耗时判断；模型文字报告不是验证结果。

In [ ]:
import json

HOST = RUN_DIR / "codex_host"
for name in ("status.json", "pending_question.json"):
  path = HOST / name
  if path.exists():
    print(name, path.read_text())
state_path = RUN_DIR / "state.json"
if state_path.exists():
  state = json.loads(state_path.read_text())
  print("Iteration:", state.get("iteration"), "/", state.get("max_iterations"))
  print("Best:", state.get("best_code_path"))
  for row in state.get("history", []):
    print(
      {
        k: row.get(k)
        for k in (
          "iteration",
          "compile_ok",
          "test_ok",
          "base_time_ms",
          "optimized_time_ms",
          "speedup",
        )
      }
    )
status = (
  json.loads((HOST / "status.json").read_text())
  if (HOST / "status.json").exists()
  else {}
)
print("原生 CLI 事件:", status.get("events"))
print("原始每轮产物:", RUN_DIR)
report = (
  Path(status["report"]) if status.get("report") else HOST / "no-report-yet"
)
if report.exists():
  from IPython.display import Markdown, display

  display(Markdown(report.read_text()))

## 7. 可选：回复人工确认后恢复
必须先打开 pending question 中引用的源码/验证产物。输入自己的批准、拒绝或修改要求；不会自动批准。正常实验不需要运行此 cell。

In [ ]:
pending = RUN_DIR / "codex_host/pending_question.json"
if pending.exists():
  print(pending.read_text())
  reply = input("阅读引用产物后，输入你的决定（留空不继续）：").strip()
  if reply:
    reply_path = RUN_DIR / "user_reply.txt"
    reply_path.write_text(reply, encoding="utf-8")
    exit_code = run_pipeline(["--reply-file", str(reply_path)])
else:
  print("没有待回复的问题。")

## 8. 可选：保存实验产物
Colab runtime 回收前下载。包含源码、对话和工具输出，请按自己的数据要求保存。原 TPU server 的队列管理见仓库 `docs/tpu-setup.md`；手动取消时使用具体 job ID，避免取消其他任务。

In [ ]:
import tarfile

from google.colab import files

archive = Path("/content") / (RUN_DIR.name + ".tar.gz")
with tarfile.open(archive, "w:gz") as tar:
  tar.add(RUN_DIR, arcname=RUN_DIR.name)
files.download(str(archive))